[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markjschmidt/PET_List/blob/main/MJS_Sonnet4_LOR_QA_Centered_20261027.ipynb)

# Vereos PET LOR Visualization — Centered QA Point Source

**Purpose:** Visualize Lines of Response (LORs) from a QA point source precisely
centered in the 676 mm FOV of a Philips Vereos PET scanner.

**Key geometry:**
- Ring diameter: 764 mm → radius R = 382 mm
- 576 crystals around ring, clockwise from 3 o'clock (crystal 0 at right)
- TOF scale: 19.5 ps/bin; BIN_TO_MM = c × T_scale / 2 = **2.923 mm/bin**
- Opposite crystals: diff = 288 (max); these LORs pass through center
- tof ≈ 0 expected for all true coincidences from a centered source

**Data source:** `short_QA_centered.txt` — 361 LOR events (357 PROMPT, 4 DELAY)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.gridspec import GridSpec
import re

# ── Vereos ring geometry ───────────────────────────────────────────────────────
N      = 576          # crystals around ring
N_MOD  = 18           # modules around ring (32 crystals each)
R_RING = 382.0        # mm  (764 mm diameter / 2)
R_FOV  = 338.0        # mm  standard reconstruction FOV
C      = 0.299792     # mm/ps  (speed of light)
T_SCALE = 19.5        # ps/bin
BIN_TO_MM = C * T_SCALE / 2   # mm per TOF bin offset from midpoint = 2.923

print(f'BIN_TO_MM = {BIN_TO_MM:.4f} mm/bin')
print(f'Ring radius = {R_RING} mm,  FOV radius = {R_FOV} mm')
print(f'Crystal spacing = {2*np.pi*R_RING/N:.2f} mm')

In [ ]:
# ── Embedded LOR data from short_QA_centered.txt ──────────────────────────────
# Format: N: PROMPT/DELAY: xa(V) xb(V) tof(V) za(V) zb(V) lea(V) leb(V)

RAW_DATA = """
2: PROMPT: xa(279) xb(568) tof(2) za(32) zb(2) lea(133) leb(122)
3: PROMPT: xa(85) xb(373) tof(1) za(11) zb(29) lea(130) leb(136)
4: PROMPT: xa(47) xb(336) tof(8) za(19) zb(19) lea(119) leb(126)
5: PROMPT: xa(138) xb(426) tof(3) za(34) zb(4) lea(130) leb(133)
6: PROMPT: xa(52) xb(515) tof(81) za(36) zb(26) lea(124) leb(133)
7: PROMPT: xa(340) xb(515) tof(86) za(3) zb(26) lea(126) leb(133)
8: PROMPT: xa(52) xb(340) tof(-5) za(36) zb(3) lea(124) leb(126)
9: PROMPT: xa(112) xb(399) tof(-9) za(24) zb(19) lea(138) leb(137)
10: PROMPT: xa(179) xb(465) tof(20) za(8) zb(31) lea(130) leb(129)
11: PROMPT: xa(22) xb(311) tof(3) za(29) zb(9) lea(130) leb(120)
12: PROMPT: xa(24) xb(313) tof(7) za(6) zb(35) lea(126) leb(125)
13: PROMPT: xa(104) xb(434) tof(-1) za(25) zb(18) lea(132) leb(126)
14: PROMPT: xa(136) xb(424) tof(-16) za(9) zb(31) lea(128) leb(128)
15: PROMPT: xa(126) xb(373) tof(-1) za(27) zb(37) lea(113) leb(128)
16: PROMPT: xa(189) xb(476) tof(-1) za(27) zb(14) lea(114) leb(120)
17: PROMPT: xa(257) xb(544) tof(-2) za(24) zb(11) lea(127) leb(126)
18: PROMPT: xa(32) xb(320) tof(9) za(16) zb(23) lea(134) leb(124)
19: PROMPT: xa(164) xb(456) tof(-17) za(13) zb(24) lea(128) leb(134)
20: PROMPT: xa(67) xb(357) tof(-15) za(8) zb(33) lea(121) leb(121)
21: PROMPT: xa(112) xb(400) tof(-1) za(3) zb(39) lea(132) leb(127)
22: PROMPT: xa(204) xb(491) tof(-6) za(15) zb(28) lea(130) leb(114)
23: PROMPT: xa(93) xb(381) tof(1) za(19) zb(23) lea(138) leb(134)
24: PROMPT: xa(278) xb(565) tof(5) za(23) zb(15) lea(145) leb(130)
25: PROMPT: xa(254) xb(538) tof(-11) za(31) zb(9) lea(125) leb(129)
26: PROMPT: xa(93) xb(381) tof(2) za(8) zb(29) lea(134) leb(122)
27: PROMPT: xa(67) xb(357) tof(-7) za(30) zb(9) lea(128) leb(131)
28: PROMPT: xa(102) xb(388) tof(-7) za(15) zb(26) lea(130) leb(126)
29: PROMPT: xa(0) xb(455) tof(-45) za(19) zb(15) lea(123) leb(133)
30: PROMPT: xa(283) xb(570) tof(-9) za(18) zb(20) lea(117) leb(127)
503: PROMPT: xa(105) xb(393) tof(10) za(13) zb(22) lea(130) leb(125)
504: PROMPT: xa(43) xb(331) tof(14) za(1) zb(39) lea(129) leb(128)
505: PROMPT: xa(35) xb(323) tof(-1) za(32) zb(9) lea(136) leb(127)
506: PROMPT: xa(31) xb(319) tof(14) za(10) zb(27) lea(128) leb(122)
507: PROMPT: xa(172) xb(465) tof(-4) za(21) zb(23) lea(126) leb(114)
508: PROMPT: xa(75) xb(362) tof(0) za(36) zb(1) lea(121) leb(124)
510: PROMPT: xa(271) xb(560) tof(0) za(29) zb(9) lea(128) leb(128)
511: PROMPT: xa(185) xb(473) tof(-9) za(2) zb(37) lea(125) leb(119)
512: PROMPT: xa(2) xb(289) tof(-5) za(3) zb(32) lea(131) leb(144)
513: PROMPT: xa(171) xb(460) tof(-6) za(24) zb(20) lea(127) leb(133)
514: PROMPT: xa(231) xb(519) tof(-15) za(23) zb(13) lea(127) leb(129)
515: PROMPT: xa(61) xb(351) tof(3) za(16) zb(24) lea(129) leb(117)
516: PROMPT: xa(187) xb(474) tof(0) za(39) zb(3) lea(134) leb(131)
517: PROMPT: xa(210) xb(499) tof(7) za(25) zb(14) lea(128) leb(124)
519: PROMPT: xa(46) xb(335) tof(-12) za(25) zb(12) lea(125) leb(116)
520: PROMPT: xa(53) xb(341) tof(-13) za(5) zb(32) lea(130) leb(129)
521: PROMPT: xa(59) xb(346) tof(1) za(11) zb(26) lea(135) leb(125)
522: PROMPT: xa(269) xb(558) tof(-7) za(37) zb(4) lea(129) leb(121)
523: PROMPT: xa(221) xb(508) tof(3) za(33) zb(7) lea(133) leb(126)
524: PROMPT: xa(73) xb(362) tof(5) za(10) zb(29) lea(134) leb(130)
525: PROMPT: xa(97) xb(384) tof(-1) za(20) zb(20) lea(121) leb(130)
526: PROMPT: xa(155) xb(443) tof(4) za(15) zb(28) lea(122) leb(134)
527: PROMPT: xa(162) xb(451) tof(15) za(19) zb(21) lea(135) leb(126)
1072: PROMPT: xa(40) xb(327) tof(0) za(12) zb(28) lea(133) leb(122)
1073: PROMPT: xa(73) xb(360) tof(-5) za(12) zb(26) lea(123) leb(116)
1074: PROMPT: xa(193) xb(481) tof(0) za(23) zb(18) lea(137) leb(127)
1075: PROMPT: xa(171) xb(458) tof(2) za(31) zb(5) lea(123) leb(129)
1076: PROMPT: xa(65) xb(354) tof(0) za(30) zb(9) lea(129) leb(133)
1077: PROMPT: xa(94) xb(381) tof(8) za(31) zb(9) lea(124) leb(122)
1078: PROMPT: xa(77) xb(365) tof(5) za(10) zb(25) lea(124) leb(136)
1080: PROMPT: xa(210) xb(498) tof(-8) za(27) zb(12) lea(129) leb(133)
1081: PROMPT: xa(3) xb(292) tof(11) za(10) zb(26) lea(135) leb(130)
1082: PROMPT: xa(42) xb(330) tof(-5) za(29) zb(11) lea(133) leb(125)
1083: PROMPT: xa(252) xb(541) tof(-2) za(33) zb(8) lea(119) leb(133)
1084: PROMPT: xa(261) xb(550) tof(3) za(19) zb(16) lea(128) leb(135)
1085: PROMPT: xa(29) xb(317) tof(-1) za(25) zb(13) lea(125) leb(132)
1086: PROMPT: xa(157) xb(444) tof(2) za(20) zb(19) lea(131) leb(125)
1087: PROMPT: xa(120) xb(407) tof(20) za(5) zb(31) lea(125) leb(134)
1088: PROMPT: xa(224) xb(543) tof(-7) za(25) zb(26) lea(120) leb(121)
1089: PROMPT: xa(162) xb(448) tof(-3) za(16) zb(24) lea(126) leb(129)
1090: PROMPT: xa(161) xb(448) tof(-6) za(23) zb(19) lea(126) leb(126)
1091: DELAY: xa(147) xb(503) tof(-77) za(16) zb(5) lea(123) leb(144)
1092: PROMPT: xa(12) xb(301) tof(3) za(39) zb(3) lea(115) leb(125)
1094: PROMPT: xa(49) xb(334) tof(9) za(5) zb(32) lea(130) leb(125)
1095: PROMPT: xa(6) xb(294) tof(5) za(1) zb(36) lea(136) leb(125)
1096: PROMPT: xa(186) xb(474) tof(9) za(13) zb(27) lea(138) leb(130)
1097: PROMPT: xa(110) xb(397) tof(4) za(12) zb(28) lea(128) leb(134)
1098: PROMPT: xa(30) xb(516) tof(0) za(21) zb(32) lea(131) leb(125)
1099: PROMPT: xa(156) xb(445) tof(-11) za(8) zb(35) lea(127) leb(126)
1100: PROMPT: xa(140) xb(426) tof(15) za(32) zb(8) lea(129) leb(127)
1101: PROMPT: xa(240) xb(527) tof(3) za(26) zb(15) lea(131) leb(126)
1649: PROMPT: xa(25) xb(303) tof(2) za(30) zb(37) lea(117) leb(118)
1650: PROMPT: xa(48) xb(337) tof(-4) za(8) zb(34) lea(128) leb(134)
1651: PROMPT: xa(262) xb(551) tof(-1) za(24) zb(19) lea(124) leb(126)
1652: PROMPT: xa(128) xb(416) tof(8) za(31) zb(8) lea(127) leb(134)
1653: PROMPT: xa(141) xb(429) tof(-6) za(30) zb(6) lea(130) leb(132)
1654: PROMPT: xa(103) xb(391) tof(4) za(15) zb(27) lea(127) leb(119)
1655: PROMPT: xa(91) xb(379) tof(16) za(20) zb(18) lea(128) leb(132)
1656: PROMPT: xa(161) xb(449) tof(-6) za(10) zb(30) lea(126) leb(124)
1657: PROMPT: xa(170) xb(458) tof(9) za(35) zb(8) lea(123) leb(124)
1658: PROMPT: xa(3) xb(291) tof(0) za(10) zb(29) lea(129) leb(134)
1659: PROMPT: xa(190) xb(479) tof(5) za(11) zb(29) lea(130) leb(144)
1661: PROMPT: xa(269) xb(557) tof(1) za(30) zb(13) lea(127) leb(128)
1662: PROMPT: xa(69) xb(428) tof(-13) za(8) zb(24) lea(137) leb(129)
1663: PROMPT: xa(148) xb(437) tof(8) za(33) zb(5) lea(114) leb(133)
1664: PROMPT: xa(280) xb(568) tof(12) za(21) zb(19) lea(135) leb(122)
1665: PROMPT: xa(127) xb(414) tof(11) za(20) zb(22) lea(138) leb(129)
1666: PROMPT: xa(52) xb(340) tof(-16) za(12) zb(31) lea(126) leb(121)
1667: PROMPT: xa(213) xb(501) tof(-2) za(9) zb(26) lea(128) leb(130)
1668: PROMPT: xa(146) xb(435) tof(-9) za(2) zb(39) lea(136) leb(131)
1669: PROMPT: xa(215) xb(501) tof(2) za(14) zb(26) lea(135) leb(131)
1670: PROMPT: xa(189) xb(478) tof(2) za(6) zb(34) lea(129) leb(116)
1671: PROMPT: xa(197) xb(485) tof(2) za(11) zb(24) lea(116) leb(132)
1672: PROMPT: xa(29) xb(318) tof(13) za(0) zb(39) lea(126) leb(127)
1674: PROMPT: xa(133) xb(418) tof(2) za(12) zb(32) lea(126) leb(129)
1675: PROMPT: xa(30) xb(318) tof(-1) za(39) zb(2) lea(125) leb(119)
1676: PROMPT: xa(267) xb(555) tof(2) za(32) zb(9) lea(123) leb(125)
1677: PROMPT: xa(273) xb(561) tof(7) za(30) zb(13) lea(128) leb(135)
1678: PROMPT: xa(208) xb(499) tof(14) za(14) zb(26) lea(129) leb(127)
2222: PROMPT: xa(76) xb(362) tof(-2) za(29) zb(12) lea(136) leb(127)
2223: PROMPT: xa(51) xb(340) tof(10) za(18) zb(17) lea(127) leb(126)
2224: PROMPT: xa(44) xb(333) tof(3) za(24) zb(12) lea(134) leb(125)
2225: PROMPT: xa(224) xb(513) tof(2) za(7) zb(33) lea(128) leb(121)
2226: PROMPT: xa(135) xb(424) tof(-7) za(36) zb(6) lea(117) leb(128)
2227: PROMPT: xa(39) xb(326) tof(4) za(22) zb(17) lea(124) leb(138)
2228: PROMPT: xa(119) xb(407) tof(0) za(28) zb(14) lea(126) leb(121)
2229: PROMPT: xa(67) xb(355) tof(-2) za(17) zb(22) lea(113) leb(124)
2230: PROMPT: xa(109) xb(397) tof(9) za(30) zb(7) lea(123) leb(122)
2232: PROMPT: xa(236) xb(523) tof(-19) za(31) zb(5) lea(126) leb(130)
2233: PROMPT: xa(99) xb(387) tof(-6) za(24) zb(14) lea(128) leb(130)
2234: PROMPT: xa(283) xb(570) tof(-8) za(36) zb(0) lea(135) leb(120)
2235: PROMPT: xa(42) xb(330) tof(-8) za(0) zb(37) lea(124) leb(130)
2236: PROMPT: xa(251) xb(538) tof(-15) za(6) zb(36) lea(132) leb(116)
2237: PROMPT: xa(109) xb(314) tof(12) za(2) zb(20) lea(120) leb(131)
2238: PROMPT: xa(26) xb(314) tof(19) za(20) zb(20) lea(139) leb(131)
2239: PROMPT: xa(170) xb(457) tof(-3) za(39) zb(1) lea(133) leb(128)
2240: PROMPT: xa(33) xb(320) tof(-6) za(13) zb(24) lea(115) leb(129)
2241: PROMPT: xa(235) xb(520) tof(-8) za(23) zb(18) lea(124) leb(120)
2242: PROMPT: xa(205) xb(495) tof(5) za(18) zb(18) lea(116) leb(124)
2243: PROMPT: xa(49) xb(337) tof(10) za(1) zb(38) lea(135) leb(132)
2244: PROMPT: xa(286) xb(574) tof(10) za(33) zb(6) lea(128) leb(124)
2245: PROMPT: xa(78) xb(366) tof(3) za(10) zb(29) lea(135) leb(134)
2247: PROMPT: xa(170) xb(459) tof(-1) za(27) zb(16) lea(123) leb(129)
2248: PROMPT: xa(57) xb(345) tof(0) za(18) zb(22) lea(131) leb(118)
2249: PROMPT: xa(79) xb(366) tof(11) za(21) zb(23) lea(121) leb(126)
2250: PROMPT: xa(65) xb(352) tof(6) za(33) zb(9) lea(138) leb(135)
2251: PROMPT: xa(249) xb(536) tof(-9) za(19) zb(17) lea(126) leb(126)
2252: PROMPT: xa(19) xb(305) tof(-2) za(37) zb(4) lea(120) leb(131)
2797: PROMPT: xa(82) xb(368) tof(1) za(24) zb(14) lea(127) leb(130)
2798: PROMPT: xa(57) xb(345) tof(-2) za(9) zb(29) lea(134) leb(132)
2799: PROMPT: xa(152) xb(440) tof(7) za(6) zb(35) lea(132) leb(122)
2800: PROMPT: xa(95) xb(381) tof(7) za(29) zb(13) lea(130) leb(134)
2801: PROMPT: xa(177) xb(464) tof(-6) za(20) zb(22) lea(132) leb(139)
2802: PROMPT: xa(196) xb(487) tof(-2) za(17) zb(23) lea(130) leb(124)
2804: PROMPT: xa(30) xb(318) tof(-16) za(18) zb(18) lea(119) leb(127)
2805: PROMPT: xa(243) xb(531) tof(1) za(32) zb(4) lea(124) leb(126)
2806: PROMPT: xa(69) xb(357) tof(-8) za(16) zb(19) lea(119) leb(129)
2807: PROMPT: xa(41) xb(329) tof(-3) za(32) zb(9) lea(122) leb(123)
2808: PROMPT: xa(234) xb(520) tof(-3) za(26) zb(12) lea(123) leb(119)
2809: PROMPT: xa(266) xb(553) tof(-14) za(27) zb(10) lea(132) leb(131)
2810: PROMPT: xa(165) xb(454) tof(0) za(24) zb(17) lea(136) leb(129)
2811: PROMPT: xa(276) xb(563) tof(-10) za(16) zb(21) lea(145) leb(131)
2812: PROMPT: xa(157) xb(440) tof(-9) za(18) zb(16) lea(135) leb(124)
2813: PROMPT: xa(226) xb(513) tof(1) za(32) zb(11) lea(124) leb(132)
2814: PROMPT: xa(172) xb(458) tof(5) za(22) zb(17) lea(133) leb(138)
2816: PROMPT: xa(30) xb(318) tof(-2) za(18) zb(25) lea(120) leb(138)
2817: PROMPT: xa(205) xb(493) tof(4) za(39) zb(0) lea(124) leb(136)
2818: PROMPT: xa(260) xb(548) tof(0) za(18) zb(22) lea(112) leb(133)
2819: PROMPT: xa(48) xb(336) tof(-4) za(13) zb(30) lea(124) leb(144)
2820: PROMPT: xa(285) xb(572) tof(-9) za(22) zb(17) lea(118) leb(117)
3361: PROMPT: xa(23) xb(309) tof(0) za(17) zb(23) lea(132) leb(136)
3362: PROMPT: xa(53) xb(341) tof(-4) za(12) zb(25) lea(138) leb(127)
3363: PROMPT: xa(235) xb(523) tof(7) za(9) zb(31) lea(125) leb(130)
3364: PROMPT: xa(12) xb(299) tof(3) za(21) zb(17) lea(123) leb(123)
3365: PROMPT: xa(147) xb(435) tof(-5) za(17) zb(19) lea(131) leb(122)
3366: PROMPT: xa(98) xb(386) tof(-6) za(8) zb(32) lea(141) leb(121)
3367: PROMPT: xa(17) xb(306) tof(-7) za(31) zb(7) lea(124) leb(122)
3368: PROMPT: xa(339) xb(454) tof(9) za(32) zb(22) lea(133) leb(136)
3369: PROMPT: xa(51) xb(454) tof(13) za(6) zb(22) lea(127) leb(136)
3370: PROMPT: xa(51) xb(339) tof(4) za(6) zb(32) lea(127) leb(133)
3371: PROMPT: xa(213) xb(499) tof(-8) za(9) zb(33) lea(129) leb(134)
3373: PROMPT: xa(69) xb(354) tof(0) za(28) zb(13) lea(135) leb(117)
3374: PROMPT: xa(129) xb(417) tof(1) za(31) zb(9) lea(119) leb(128)
3375: PROMPT: xa(229) xb(516) tof(-12) za(18) zb(19) lea(134) leb(134)
3376: PROMPT: xa(151) xb(439) tof(10) za(11) zb(30) lea(134) leb(125)
3377: PROMPT: xa(187) xb(475) tof(-6) za(26) zb(16) lea(128) leb(132)
3378: PROMPT: xa(264) xb(550) tof(2) za(28) zb(10) lea(125) leb(113)
3379: PROMPT: xa(144) xb(432) tof(2) za(29) zb(11) lea(120) leb(136)
3380: PROMPT: xa(144) xb(433) tof(-8) za(7) zb(36) lea(132) leb(130)
3381: PROMPT: xa(108) xb(400) tof(-6) za(24) zb(9) lea(132) leb(130)
3382: PROMPT: xa(50) xb(340) tof(1) za(14) zb(26) lea(115) leb(127)
3383: PROMPT: xa(139) xb(427) tof(2) za(3) zb(36) lea(138) leb(138)
3384: PROMPT: xa(53) xb(341) tof(8) za(17) zb(26) lea(132) leb(134)
3385: PROMPT: xa(198) xb(486) tof(-2) za(29) zb(12) lea(134) leb(138)
3386: PROMPT: xa(206) xb(494) tof(-2) za(28) zb(8) lea(137) leb(123)
3387: PROMPT: xa(82) xb(369) tof(14) za(23) zb(14) lea(116) leb(120)
3388: PROMPT: xa(96) xb(384) tof(7) za(15) zb(21) lea(123) leb(125)
3390: PROMPT: xa(16) xb(304) tof(-3) za(17) zb(25) lea(131) leb(124)
3391: PROMPT: xa(131) xb(418) tof(9) za(31) zb(5) lea(128) leb(139)
3392: PROMPT: xa(115) xb(412) tof(11) za(20) zb(36) lea(121) leb(122)
3393: PROMPT: xa(88) xb(376) tof(3) za(7) zb(31) lea(123) leb(135)
3394: PROMPT: xa(151) xb(439) tof(0) za(4) zb(37) lea(121) leb(135)
3940: PROMPT: xa(91) xb(379) tof(-2) za(7) zb(35) lea(124) leb(120)
3941: PROMPT: xa(225) xb(512) tof(12) za(15) zb(21) lea(133) leb(128)
3942: PROMPT: xa(280) xb(569) tof(0) za(9) zb(34) lea(119) leb(124)
3943: PROMPT: xa(24) xb(313) tof(-1) za(9) zb(27) lea(138) leb(130)
3945: PROMPT: xa(262) xb(553) tof(5) za(34) zb(6) lea(128) leb(126)
3946: PROMPT: xa(109) xb(396) tof(-8) za(27) zb(12) lea(131) leb(126)
3947: PROMPT: xa(123) xb(411) tof(-9) za(24) zb(13) lea(135) leb(127)
3948: PROMPT: xa(124) xb(411) tof(8) za(14) zb(24) lea(138) leb(132)
3949: PROMPT: xa(172) xb(460) tof(8) za(16) zb(23) lea(131) leb(126)
3950: PROMPT: xa(179) xb(467) tof(-2) za(28) zb(11) lea(136) leb(138)
3951: PROMPT: xa(98) xb(387) tof(6) za(22) zb(17) lea(129) leb(127)
3952: PROMPT: xa(128) xb(416) tof(-1) za(36) zb(3) lea(132) leb(138)
3953: PROMPT: xa(3) xb(290) tof(2) za(24) zb(14) lea(133) leb(133)
3954: PROMPT: xa(85) xb(373) tof(6) za(30) zb(8) lea(124) leb(121)
3955: PROMPT: xa(102) xb(390) tof(-1) za(18) zb(18) lea(131) leb(117)
3956: PROMPT: xa(230) xb(518) tof(7) za(20) zb(21) lea(129) leb(129)
3958: PROMPT: xa(66) xb(354) tof(-7) za(12) zb(30) lea(132) leb(130)
3959: PROMPT: xa(46) xb(334) tof(-3) za(33) zb(7) lea(127) leb(125)
3960: PROMPT: xa(198) xb(486) tof(3) za(32) zb(6) lea(132) leb(129)
3961: PROMPT: xa(55) xb(343) tof(-1) za(36) zb(3) lea(118) leb(139)
3962: PROMPT: xa(79) xb(366) tof(-3) za(7) zb(37) lea(134) leb(127)
3963: PROMPT: xa(86) xb(377) tof(-9) za(11) zb(30) lea(125) leb(130)
3964: PROMPT: xa(234) xb(520) tof(16) za(18) zb(21) lea(123) leb(125)
3965: PROMPT: xa(216) xb(506) tof(6) za(24) zb(17) lea(127) leb(136)
3966: PROMPT: xa(287) xb(575) tof(17) za(34) zb(2) lea(132) leb(123)
3967: PROMPT: xa(24) xb(311) tof(15) za(13) zb(27) lea(124) leb(131)
4512: PROMPT: xa(122) xb(410) tof(-2) za(26) zb(15) lea(134) leb(124)
4513: PROMPT: xa(265) xb(553) tof(0) za(38) zb(2) lea(127) leb(128)
4514: PROMPT: xa(258) xb(546) tof(13) za(24) zb(17) lea(127) leb(132)
4515: PROMPT: xa(30) xb(318) tof(0) za(31) zb(11) lea(123) leb(130)
4516: PROMPT: xa(43) xb(332) tof(-1) za(19) zb(20) lea(133) leb(128)
4517: PROMPT: xa(85) xb(373) tof(-8) za(18) zb(25) lea(137) leb(125)
4518: PROMPT: xa(242) xb(530) tof(-4) za(15) zb(26) lea(132) leb(128)
4519: PROMPT: xa(13) xb(299) tof(4) za(0) zb(38) lea(136) leb(128)
4520: PROMPT: xa(171) xb(458) tof(5) za(18) zb(23) lea(131) leb(135)
4521: PROMPT: xa(272) xb(560) tof(-12) za(39) zb(5) lea(123) leb(123)
4523: PROMPT: xa(93) xb(381) tof(5) za(13) zb(26) lea(115) leb(123)
4524: DELAY: xa(43) xb(484) tof(71) za(19) zb(3) lea(133) leb(127)
4525: DELAY: xa(332) xb(484) tof(72) za(20) zb(3) lea(128) leb(127)
4526: PROMPT: xa(152) xb(443) tof(0) za(17) zb(25) lea(123) leb(122)
4527: PROMPT: xa(53) xb(342) tof(-12) za(32) zb(7) lea(131) leb(120)
4528: PROMPT: xa(125) xb(413) tof(-5) za(33) zb(7) lea(135) leb(131)
4529: PROMPT: xa(186) xb(473) tof(-9) za(29) zb(12) lea(127) leb(121)
4530: PROMPT: xa(205) xb(493) tof(-3) za(37) zb(6) lea(139) leb(127)
4531: PROMPT: xa(80) xb(369) tof(-6) za(32) zb(10) lea(118) leb(127)
4532: PROMPT: xa(45) xb(333) tof(-16) za(18) zb(24) lea(131) leb(138)
4533: PROMPT: xa(74) xb(362) tof(-1) za(21) zb(20) lea(129) leb(137)
4534: PROMPT: xa(99) xb(386) tof(-5) za(3) zb(36) lea(115) leb(125)
4535: PROMPT: xa(123) xb(410) tof(-7) za(27) zb(12) lea(132) leb(130)
4536: PROMPT: xa(237) xb(522) tof(-5) za(37) zb(4) lea(127) leb(133)
4537: PROMPT: xa(37) xb(325) tof(-8) za(7) zb(31) lea(116) leb(113)
4538: PROMPT: xa(143) xb(433) tof(-8) za(6) zb(38) lea(113) leb(115)
4540: PROMPT: xa(287) xb(573) tof(-3) za(33) zb(3) lea(126) leb(135)
4541: PROMPT: xa(225) xb(512) tof(-3) za(38) zb(2) lea(135) leb(125)
4542: PROMPT: xa(26) xb(313) tof(-2) za(18) zb(23) lea(122) leb(138)
4543: PROMPT: xa(145) xb(432) tof(1) za(2) zb(33) lea(130) leb(118)
4544: PROMPT: xa(208) xb(496) tof(7) za(33) zb(10) lea(126) leb(132)
4545: PROMPT: xa(264) xb(552) tof(-1) za(17) zb(24) lea(132) leb(124)
5092: PROMPT: xa(69) xb(357) tof(4) za(14) zb(27) lea(126) leb(123)
5093: PROMPT: xa(189) xb(476) tof(-15) za(31) zb(11) lea(126) leb(135)
5094: PROMPT: xa(266) xb(551) tof(10) za(25) zb(14) lea(117) leb(130)
5095: PROMPT: xa(218) xb(505) tof(-10) za(7) zb(31) lea(123) leb(127)
5096: PROMPT: xa(77) xb(364) tof(-8) za(30) zb(8) lea(123) leb(132)
5097: PROMPT: xa(149) xb(437) tof(1) za(21) zb(14) lea(118) leb(130)
5098: PROMPT: xa(277) xb(562) tof(5) za(23) zb(14) lea(132) leb(131)
5099: PROMPT: xa(244) xb(532) tof(-19) za(13) zb(27) lea(133) leb(121)
5100: PROMPT: xa(115) xb(403) tof(-3) za(13) zb(26) lea(129) leb(127)
5101: PROMPT: xa(226) xb(514) tof(-7) za(23) zb(14) lea(127) leb(132)
5102: PROMPT: xa(129) xb(418) tof(-8) za(19) zb(17) lea(134) leb(125)
5104: PROMPT: xa(46) xb(334) tof(3) za(17) zb(24) lea(136) leb(133)
5105: PROMPT: xa(273) xb(561) tof(0) za(25) zb(16) lea(133) leb(123)
5106: PROMPT: xa(277) xb(565) tof(-5) za(15) zb(26) lea(137) leb(123)
5107: PROMPT: xa(151) xb(438) tof(4) za(11) zb(24) lea(128) leb(133)
5108: PROMPT: xa(137) xb(423) tof(7) za(12) zb(27) lea(127) leb(125)
5109: PROMPT: xa(97) xb(385) tof(2) za(6) zb(36) lea(125) leb(130)
5110: PROMPT: xa(126) xb(414) tof(-3) za(13) zb(30) lea(127) leb(123)
5111: PROMPT: xa(248) xb(535) tof(8) za(32) zb(10) lea(130) leb(131)
5112: PROMPT: xa(168) xb(458) tof(-13) za(31) zb(7) lea(126) leb(122)
5113: PROMPT: xa(147) xb(434) tof(-1) za(39) zb(3) lea(128) leb(118)
5114: PROMPT: xa(25) xb(322) tof(8) za(36) zb(19) lea(127) leb(126)
5115: PROMPT: xa(30) xb(318) tof(17) za(0) zb(39) lea(124) leb(125)
5116: PROMPT: xa(244) xb(531) tof(10) za(14) zb(27) lea(130) leb(129)
5117: PROMPT: xa(22) xb(311) tof(1) za(6) zb(38) lea(117) leb(124)
5118: PROMPT: xa(185) xb(474) tof(7) za(35) zb(4) lea(122) leb(133)
5119: PROMPT: xa(90) xb(379) tof(7) za(39) zb(0) lea(126) leb(119)
5120: PROMPT: xa(97) xb(385) tof(5) za(3) zb(31) lea(128) leb(120)
5121: PROMPT: xa(280) xb(568) tof(0) za(12) zb(24) lea(125) leb(132)
5123: PROMPT: xa(276) xb(564) tof(3) za(20) zb(20) lea(117) leb(118)
5124: PROMPT: xa(26) xb(314) tof(-8) za(25) zb(17) lea(125) leb(120)
5125: PROMPT: xa(188) xb(475) tof(-15) za(35) zb(0) lea(125) leb(125)
5126: PROMPT: xa(272) xb(560) tof(-3) za(31) zb(7) lea(134) leb(128)
5127: PROMPT: xa(9) xb(297) tof(14) za(36) zb(0) lea(126) leb(125)
5128: PROMPT: xa(199) xb(487) tof(3) za(36) zb(4) lea(123) leb(118)
5129: PROMPT: xa(49) xb(337) tof(9) za(28) zb(11) lea(126) leb(120)
5130: PROMPT: xa(240) xb(528) tof(12) za(5) zb(33) lea(130) leb(127)
5131: PROMPT: xa(202) xb(489) tof(2) za(26) zb(16) lea(124) leb(129)
5676: PROMPT: xa(193) xb(481) tof(5) za(33) zb(10) lea(129) leb(122)
5677: PROMPT: xa(79) xb(367) tof(-15) za(32) zb(12) lea(116) leb(133)
5678: PROMPT: xa(83) xb(371) tof(3) za(20) zb(16) lea(132) leb(121)
5679: PROMPT: xa(244) xb(533) tof(-10) za(10) zb(29) lea(122) leb(123)
5680: PROMPT: xa(98) xb(385) tof(-2) za(2) zb(34) lea(132) leb(118)
5682: PROMPT: xa(258) xb(547) tof(15) za(14) zb(27) lea(125) leb(131)
5683: PROMPT: xa(73) xb(361) tof(8) za(7) zb(32) lea(123) leb(130)
5684: PROMPT: xa(1) xb(289) tof(12) za(5) zb(32) lea(127) leb(125)
5685: PROMPT: xa(266) xb(552) tof(3) za(28) zb(13) lea(127) leb(140)
5686: PROMPT: xa(130) xb(418) tof(-6) za(26) zb(11) lea(128) leb(129)
5687: PROMPT: xa(83) xb(370) tof(6) za(37) zb(5) lea(134) leb(135)
5688: PROMPT: xa(69) xb(358) tof(6) za(34) zb(1) lea(119) leb(130)
5689: PROMPT: xa(44) xb(329) tof(0) za(3) zb(32) lea(124) leb(127)
5690: PROMPT: xa(57) xb(345) tof(-13) za(12) zb(31) lea(123) leb(116)
5692: PROMPT: xa(276) xb(563) tof(9) za(15) zb(23) lea(134) leb(133)
5693: PROMPT: xa(193) xb(480) tof(-10) za(34) zb(7) lea(141) leb(118)
5694: PROMPT: xa(243) xb(531) tof(-17) za(8) zb(28) lea(114) leb(135)
5695: PROMPT: xa(185) xb(472) tof(-1) za(26) zb(10) lea(127) leb(127)
5696: PROMPT: xa(99) xb(387) tof(2) za(38) zb(1) lea(127) leb(133)
5697: PROMPT: xa(151) xb(438) tof(7) za(21) zb(20) lea(122) leb(129)
5698: PROMPT: xa(255) xb(542) tof(12) za(32) zb(9) lea(129) leb(125)
5699: PROMPT: xa(78) xb(366) tof(2) za(38) zb(3) lea(143) leb(124)
6244: PROMPT: xa(12) xb(347) tof(-13) za(39) zb(18) lea(114) leb(134)
6245: PROMPT: xa(59) xb(347) tof(0) za(26) zb(18) lea(130) leb(134)
6246: PROMPT: xa(5) xb(293) tof(6) za(30) zb(8) lea(126) leb(117)
6247: PROMPT: xa(212) xb(500) tof(1) za(23) zb(14) lea(122) leb(130)
6248: PROMPT: xa(267) xb(554) tof(3) za(30) zb(12) lea(129) leb(135)
6249: PROMPT: xa(206) xb(493) tof(-12) za(30) zb(6) lea(131) leb(129)
6250: PROMPT: xa(24) xb(311) tof(6) za(21) zb(15) lea(126) leb(123)
6251: PROMPT: xa(77) xb(366) tof(4) za(30) zb(9) lea(113) leb(117)
6252: PROMPT: xa(119) xb(406) tof(-1) za(36) zb(1) lea(137) leb(137)
6253: PROMPT: xa(163) xb(450) tof(-13) za(8) zb(31) lea(125) leb(123)
6254: PROMPT: xa(19) xb(308) tof(4) za(8) zb(33) lea(129) leb(134)
6255: PROMPT: xa(166) xb(452) tof(25) za(11) zb(27) lea(129) leb(143)
6256: PROMPT: xa(67) xb(357) tof(-4) za(10) zb(29) lea(129) leb(131)
6257: PROMPT: xa(233) xb(523) tof(6) za(35) zb(5) lea(131) leb(132)
6258: PROMPT: xa(256) xb(544) tof(-22) za(14) zb(22) lea(131) leb(124)
6260: PROMPT: xa(41) xb(332) tof(-7) za(10) zb(24) lea(141) leb(128)
6261: PROMPT: xa(38) xb(326) tof(-8) za(24) zb(15) lea(114) leb(126)
6262: PROMPT: xa(65) xb(353) tof(5) za(8) zb(34) lea(137) leb(120)
6263: PROMPT: xa(23) xb(311) tof(8) za(23) zb(16) lea(124) leb(130)
6264: PROMPT: xa(252) xb(541) tof(0) za(8) zb(34) lea(130) leb(131)
6265: PROMPT: xa(249) xb(536) tof(2) za(35) zb(1) lea(130) leb(134)
6266: PROMPT: xa(102) xb(390) tof(10) za(4) zb(35) lea(124) leb(120)
6267: PROMPT: xa(97) xb(384) tof(11) za(6) zb(35) lea(116) leb(138)
6268: PROMPT: xa(212) xb(497) tof(-16) za(18) zb(21) lea(129) leb(126)
6269: PROMPT: xa(226) xb(515) tof(2) za(24) zb(11) lea(134) leb(124)
6270: PROMPT: xa(97) xb(386) tof(1) za(24) zb(15) lea(136) leb(125)
6817: PROMPT: xa(93) xb(381) tof(-1) za(38) zb(4) lea(120) leb(126)
6818: PROMPT: xa(223) xb(510) tof(-15) za(22) zb(19) lea(124) leb(129)
6819: PROMPT: xa(270) xb(560) tof(-10) za(15) zb(27) lea(123) leb(121)
6820: PROMPT: xa(104) xb(392) tof(-6) za(35) zb(7) lea(117) leb(126)
6821: PROMPT: xa(81) xb(344) tof(5) za(26) zb(0) lea(132) leb(128)
6822: PROMPT: xa(161) xb(446) tof(0) za(16) zb(26) lea(133) leb(138)
6823: PROMPT: xa(65) xb(352) tof(-10) za(38) zb(4) lea(130) leb(120)
6824: PROMPT: xa(225) xb(513) tof(-1) za(20) zb(23) lea(136) leb(127)
6825: PROMPT: xa(185) xb(473) tof(1) za(13) zb(29) lea(138) leb(135)
6826: PROMPT: xa(140) xb(428) tof(-7) za(1) zb(35) lea(122) leb(133)
6827: PROMPT: xa(190) xb(477) tof(1) za(27) zb(9) lea(123) leb(128)
6828: PROMPT: xa(112) xb(403) tof(-9) za(16) zb(21) lea(129) leb(127)
6829: DELAY: xa(185) xb(425) tof(-52) za(13) zb(35) lea(138) leb(121)
6830: PROMPT: xa(21) xb(309) tof(-17) za(26) zb(15) lea(112) leb(130)
6832: PROMPT: xa(284) xb(569) tof(7) za(9) zb(28) lea(132) leb(127)
6833: PROMPT: xa(92) xb(380) tof(-3) za(8) zb(30) lea(133) leb(126)
6834: PROMPT: xa(1) xb(289) tof(-8) za(9) zb(27) lea(139) leb(114)
6835: PROMPT: xa(199) xb(485) tof(5) za(12) zb(29) lea(132) leb(136)
6836: PROMPT: xa(183) xb(470) tof(-1) za(20) zb(18) lea(128) leb(129)
6837: PROMPT: xa(42) xb(330) tof(-11) za(9) zb(29) lea(134) leb(127)
6838: PROMPT: xa(238) xb(525) tof(2) za(38) zb(2) lea(130) leb(122)
6839: PROMPT: xa(104) xb(391) tof(3) za(28) zb(9) lea(125) leb(130)
6840: PROMPT: xa(278) xb(566) tof(25) za(17) zb(20) lea(122) leb(123)
6841: PROMPT: xa(238) xb(526) tof(0) za(3) zb(33) lea(131) leb(132)
6842: PROMPT: xa(189) xb(476) tof(-7) za(14) zb(29) lea(128) leb(126)
6843: PROMPT: xa(28) xb(317) tof(9) za(4) zb(37) lea(134) leb(133)
"""

def parse_lor_data(text):
    """Parse LOR data from listPrint text format."""
    events = []
    pattern = re.compile(
        r'(PROMPT|DELAY):\s*xa\((-?\d+)\)\s+xb\((-?\d+)\)\s+tof\((-?\d+)\)'
    )
    for m in pattern.finditer(text):
        etype, xa, xb, tof = m.group(1), int(m.group(2)), int(m.group(3)), int(m.group(4))
        diff = min(abs(xb - xa), N - abs(xb - xa))
        events.append({'type': etype, 'xa': xa, 'xb': xb, 'tof': tof, 'diff': diff})
    return events

events = parse_lor_data(RAW_DATA)
n_prompt = sum(1 for e in events if e['type'] == 'PROMPT')
n_delay  = sum(1 for e in events if e['type'] == 'DELAY')
print(f'Parsed {len(events)} events: {n_prompt} PROMPT, {n_delay} DELAY')

diffs = [e['diff'] for e in events]
tofs  = [e['tof']  for e in events]
print(f'crystal_diff: min={min(diffs)}, max={max(diffs)}, mean={np.mean(diffs):.1f}')
print(f'tof range: {min(tofs)} to {max(tofs)} bins')

# Flag events with diff far from 288 (short-chord / non-opposite LORs)
n_short = sum(1 for e in events if e['diff'] < 240)
print(f'Events with crystal_diff < 240 (short-chord): {n_short}')

In [ ]:
# ── Geometry helper functions ──────────────────────────────────────────────────

def crystal_xy(k):
    """2D position (mm) of crystal k on the Vereos ring.
    Crystal 0 = 3 o'clock (positive X). Clockwise = increasing k."""
    theta = -2 * np.pi * k / N          # negative → clockwise in standard coords
    return R_RING * np.cos(theta), R_RING * np.sin(theta)

def tof_position(xa, xb, tof_bins):
    """TOF-estimated 2D annihilation position.

    For tof = t_a - t_b  (time at crystal A minus time at crystal B):
      tof > 0  →  photon at A arrived later  →  source closer to B
      offset = +tof * BIN_TO_MM in direction A→B from midpoint
    """
    ax, ay = crystal_xy(xa)
    bx, by = crystal_xy(xb)
    mx, my = (ax + bx) / 2.0, (ay + by) / 2.0          # LOR midpoint
    dx, dy = bx - ax, by - ay
    L = np.hypot(dx, dy)
    dx, dy = dx / L, dy / L                              # unit vector A→B
    px = mx + tof_bins * BIN_TO_MM * dx
    py = my + tof_bins * BIN_TO_MM * dy
    return px, py

# Pre-compute all crystal positions for plotting
cx_all = np.array([crystal_xy(k)[0] for k in range(N)])
cy_all = np.array([crystal_xy(k)[1] for k in range(N)])

# Verify crystal 0 at 3 o'clock and crystal 144 at 6 o'clock
print(f'Crystal   0: ({cx_all[0]:.1f}, {cy_all[0]:.1f}) mm  ← should be ({R_RING:.0f}, 0)')
print(f'Crystal 144: ({cx_all[144]:.1f}, {cy_all[144]:.1f}) mm  ← should be (0, {-R_RING:.0f})')
print(f'Crystal 288: ({cx_all[288]:.1f}, {cy_all[288]:.1f}) mm  ← should be ({-R_RING:.0f}, 0)')

In [ ]:
# ── Main visualization: Ring LOR view + TOF positions ─────────────────────────

fig = plt.figure(figsize=(18, 8))
gs  = GridSpec(1, 2, figure=fig, width_ratios=[1.1, 1], wspace=0.35)
ax1 = fig.add_subplot(gs[0])   # full ring + LOR lines
ax2 = fig.add_subplot(gs[1])   # TOF-estimated positions (zoom)

# ── Color maps ────────────────────────────────────────────────────────────────
NORM_TOF  = Normalize(vmin=0, vmax=25)
CMAP_TOF  = plt.cm.YlOrRd        # yellow (small tof) → red (large tof)

# ── Panel 1: Ring view ────────────────────────────────────────────────────────

# Draw PET ring
ring_theta = np.linspace(0, 2*np.pi, 1000)
ax1.plot(R_RING * np.cos(ring_theta), R_RING * np.sin(ring_theta),
         color='#888', lw=1.0, alpha=0.6, zorder=1)

# Reconstruction FOV circle
ax1.plot(R_FOV * np.cos(ring_theta), R_FOV * np.sin(ring_theta),
         color='steelblue', lw=0.8, ls='--', alpha=0.5, zorder=1,
         label=f'Recon FOV (R={R_FOV:.0f} mm)')

# All crystal positions (tiny dots)
ax1.scatter(cx_all, cy_all, s=1.5, color='#aaa', alpha=0.5, zorder=2)

# Module boundary tick marks (18 modules, 32 crystals each)
CPM = N // N_MOD
for m in range(N_MOD):
    k = m * CPM
    cx, cy = crystal_xy(k)
    ax1.plot([0.90 * cx, 1.06 * cx], [0.90 * cy, 1.06 * cy],
             color='#555', lw=1.2, alpha=0.7, zorder=3)
    ax1.text(1.10 * cx, 1.10 * cy, str(m),
             ha='center', va='center', fontsize=6, color='#444')

# Draw all LOR lines
for ev in events:
    ax_c, ay_c = crystal_xy(ev['xa'])
    bx_c, by_c = crystal_xy(ev['xb'])
    if ev['type'] == 'DELAY':
        ax1.plot([ax_c, bx_c], [ay_c, by_c],
                 color='blue', lw=1.2, alpha=0.7, zorder=5)
    elif ev['diff'] < 240:
        # Short-chord LOR (non-opposite crystals) — scatter/oblique
        ax1.plot([ax_c, bx_c], [ay_c, by_c],
                 color='purple', lw=0.8, alpha=0.5, zorder=4)
    else:
        # Normal opposite-crystal LOR colored by |tof|
        color = CMAP_TOF(NORM_TOF(abs(ev['tof'])))
        ax1.plot([ax_c, bx_c], [ay_c, by_c],
                 color=color, lw=0.5, alpha=0.35, zorder=4)

# Mark TOF-estimated annihilation positions
for ev in events:
    px, py = tof_position(ev['xa'], ev['xb'], ev['tof'])
    if ev['type'] == 'DELAY':
        ax1.plot(px, py, 'b^', ms=7, mew=1.5, alpha=0.9, zorder=8)
    else:
        ax1.plot(px, py, '.', color=CMAP_TOF(NORM_TOF(abs(ev['tof']))),
                 ms=4, alpha=0.7, zorder=7)

# Center cross
ax1.plot(0, 0, 'k+', ms=14, mew=2.0, zorder=10, label='FOV center (0,0)')

# Crystal 0 label
ax1.annotate('Crystal 0\n(3 o\'clock)', xy=(R_RING+5, 0),
             fontsize=7, color='#444', va='center')

# Colorbar for TOF
sm = ScalarMappable(cmap=CMAP_TOF, norm=NORM_TOF)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax1, fraction=0.03, pad=0.02)
cbar.set_label('|TOF| bins', fontsize=9)

ax1.set_xlim(-470, 490)
ax1.set_ylim(-460, 460)
ax1.set_aspect('equal')
ax1.set_title(f'Vereos Ring — All {len(events)} LORs\n'
              f'(PROMPT={n_prompt} green-red | DELAY={n_delay} blue | '
              f'short-chord={n_short} purple)', fontsize=10)
ax1.set_xlabel('X (mm)  [crystal 0 at right]', fontsize=9)
ax1.set_ylabel('Y (mm)', fontsize=9)
ax1.legend(fontsize=8, loc='upper left')
ax1.grid(True, alpha=0.15)

# ── Panel 2: TOF-estimated positions, zoomed ─────────────────────────────────

ZOOM = 120   # mm — zoom radius for center view

# FOV circle at zoom scale
ax2.plot(R_FOV * np.cos(ring_theta), R_FOV * np.sin(ring_theta),
         color='steelblue', lw=0.8, ls='--', alpha=0.4)

prompt_xs, prompt_ys, prompt_tofs_abs = [], [], []
delay_xs,  delay_ys  = [], []

for ev in events:
    px, py = tof_position(ev['xa'], ev['xb'], ev['tof'])
    if ev['type'] == 'DELAY':
        delay_xs.append(px)
        delay_ys.append(py)
    else:
        prompt_xs.append(px)
        prompt_ys.append(py)
        prompt_tofs_abs.append(abs(ev['tof']))

sc = ax2.scatter(prompt_xs, prompt_ys, c=prompt_tofs_abs,
                 cmap=CMAP_TOF, norm=NORM_TOF,
                 s=18, alpha=0.75, zorder=4, label='PROMPT')
ax2.scatter(delay_xs, delay_ys, color='blue', s=60,
            marker='^', zorder=6, label=f'DELAY ({n_delay})')

# Crosshairs at true center
ax2.axhline(0, color='#555', lw=0.6, alpha=0.5)
ax2.axvline(0, color='#555', lw=0.6, alpha=0.5)
ax2.plot(0, 0, 'k+', ms=16, mew=2.5, zorder=8, label='True center')

# TOF resolution circle (320 ps FWHM → ±48 mm sigma ≈ FWHM/2.35)
tof_fwhm_mm = 0.320 * C / 2 * 1000   # ns → mm FWHM
tof_sigma   = tof_fwhm_mm / 2.355
circle_res  = plt.Circle((0, 0), tof_fwhm_mm/2, fill=False,
                          color='gray', ls=':', lw=1, alpha=0.7)
ax2.add_patch(circle_res)
ax2.text(tof_fwhm_mm/2+2, 2, f'TOF FWHM\n({tof_fwhm_mm:.0f} mm)',
         fontsize=7, color='gray')

cb2 = fig.colorbar(sc, ax=ax2, fraction=0.04, pad=0.02)
cb2.set_label('|TOF| bins', fontsize=9)

ax2.set_xlim(-ZOOM, ZOOM)
ax2.set_ylim(-ZOOM, ZOOM)
ax2.set_aspect('equal')
ax2.set_title('TOF-Estimated Annihilation Positions\n(zoomed ±120 mm)', fontsize=10)
ax2.set_xlabel('X (mm)', fontsize=9)
ax2.set_ylabel('Y (mm)', fontsize=9)
ax2.legend(fontsize=8, loc='upper right')
ax2.grid(True, alpha=0.2)

fig.suptitle('QA Point Source — Centered in 676 mm FOV\n'
             f'Vereos Ring: R={R_RING:.0f} mm, N={N} crystals,  '
             f'TOF scale={T_SCALE} ps/bin,  BIN_TO_MM={BIN_TO_MM:.3f} mm/bin',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('LOR_QA_Centered.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: LOR_QA_Centered.png')

In [ ]:
# ── Summary statistics and TOF histogram ──────────────────────────────────────

import warnings
warnings.filterwarnings('ignore')

prompt_events = [e for e in events if e['type'] == 'PROMPT']
delay_events  = [e for e in events if e['type'] == 'DELAY']

# Radial distance of each reconstructed position from center
radii = []
for ev in prompt_events:
    px, py = tof_position(ev['xa'], ev['xb'], ev['tof'])
    radii.append(np.hypot(px, py))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) TOF distribution
ptofs = [e['tof'] for e in prompt_events]
dtofs = [e['tof'] for e in delay_events]
axes[0].hist(ptofs, bins=30, color='tomato', alpha=0.8, label='PROMPT')
axes[0].hist(dtofs, bins=10, color='royalblue', alpha=0.8, label='DELAY')
axes[0].axvline(0, color='k', lw=1.5, ls='--')
axes[0].set_xlabel('TOF bin value', fontsize=10)
axes[0].set_ylabel('Count', fontsize=10)
axes[0].set_title('TOF Distribution', fontsize=10)
axes[0].legend()
axes[0].text(0.98, 0.95,
             f'PROMPT: mean={np.mean(ptofs):.1f}, σ={np.std(ptofs):.1f}\n'
             f'       ({np.mean(ptofs)*BIN_TO_MM:.1f} ± {np.std(ptofs)*BIN_TO_MM:.1f} mm)',
             transform=axes[0].transAxes, ha='right', va='top',
             fontsize=8, bbox=dict(boxstyle='round', fc='white', alpha=0.7))

# 2) Crystal diff distribution
p_diffs = [e['diff'] for e in prompt_events]
d_diffs = [e['diff'] for e in delay_events]
axes[1].hist(p_diffs, bins=30, color='tomato', alpha=0.8, label='PROMPT')
axes[1].hist(d_diffs, bins=10, color='royalblue', alpha=0.8, label='DELAY')
axes[1].axvline(288, color='k', lw=1.5, ls='--', label='Max (288)')
axes[1].set_xlabel('Crystal diff  |xa − xb| mod 576', fontsize=10)
axes[1].set_ylabel('Count', fontsize=10)
axes[1].set_title('Crystal Separation', fontsize=10)
axes[1].legend(fontsize=8)

# 3) Radial distance of reconstructed positions
axes[2].hist(radii, bins=25, color='mediumseagreen', alpha=0.85, edgecolor='white')
axes[2].axvline(np.mean(radii), color='k', lw=1.5, ls='--',
               label=f'Mean={np.mean(radii):.1f} mm')
axes[2].set_xlabel('Radial distance from center (mm)', fontsize=10)
axes[2].set_ylabel('Count', fontsize=10)
axes[2].set_title('PROMPT: |Reconstructed Position|', fontsize=10)
axes[2].legend(fontsize=9)
axes[2].text(0.98, 0.95,
             f'Mean = {np.mean(radii):.1f} mm\nMedian = {np.median(radii):.1f} mm',
             transform=axes[2].transAxes, ha='right', va='top',
             fontsize=9, bbox=dict(boxstyle='round', fc='white', alpha=0.7))

fig.suptitle('QA Centered Point Source — Event Statistics', fontsize=11)
plt.tight_layout()
plt.savefig('LOR_QA_Centered_stats.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n── Summary ──────────────────────────────────────────────────────')
print(f'Total events:          {len(events)}')
print(f'  PROMPT:              {n_prompt}  ({100*n_prompt/len(events):.1f}%)')
print(f'  DELAY (randoms):     {n_delay}   ({100*n_delay/len(events):.1f}%)')
print(f'PROMPT TOF mean:       {np.mean(ptofs):.2f} bins ({np.mean(ptofs)*BIN_TO_MM:.2f} mm offset)')
print(f'PROMPT TOF std:        {np.std(ptofs):.2f} bins ({np.std(ptofs)*BIN_TO_MM:.2f} mm)')
print(f'DELAY TOF values:      {sorted(dtofs)}  (randoms — large |tof|)')
print(f'Events with diff<240:  {n_short}  (short-chord / scatter / oblique LORs)')
print(f'Radial pos mean:       {np.mean(radii):.1f} mm  (ideal = 0 for perfectly centered source)')
print(f'Vereos TOF FWHM:       320 ps → {tof_fwhm_mm:.1f} mm spatial resolution')

## How to Load Your Own Data File

Replace the `RAW_DATA` string in Cell 3 with a file upload:

```python
from google.colab import files
uploaded = files.upload()   # choose short_QA_centered.txt
fname = list(uploaded.keys())[0]
file_text = uploaded[fname].decode('utf-8')
events = parse_lor_data(file_text)
```

Or mount Google Drive:
```python
from google.colab import drive
drive.mount('/content/drive')
with open('/content/drive/MyDrive/path/to/short_QA_centered.txt') as f:
    events = parse_lor_data(f.read())
```

## Physics notes

| Quantity | Value | Formula |
|---|---|---|
| Ring radius | 382 mm | 764 mm / 2 |
| Crystal spacing | 4.17 mm | 2π × 382 / 576 |
| TOF bin scale | 19.5 ps/bin | from Vereos header |
| BIN_TO_MM | 2.923 mm/bin | c × 19.5 ps / 2 |
| Max crystal diff | 288 | N/2 = 576/2 |
| TOF FWHM (320 ps) | ≈48 mm | c × 0.320 ns / 2 |

**TOF sign convention:** `tof = t_a – t_b`.  
`tof > 0` → crystal A detected its photon later → annihilation point is closer to B.  
Reconstructed position: **midpoint + tof × BIN_TO_MM × unit(A→B)**